In [1]:
import pandas as pd
import json
from glob import glob
from new_module.dev_utils.utils import read_outputs

### Toxicity (Save index)

In [ ]:
# list of run ids
run_ids = ["vzdu2wd8", 
           "wwhu40zw",
           "4hp0i0eu",
           "9ldaxvli",
           "ny9pw5ce",
           "48bqsqpf"]

In [20]:
common_edited_indexes = set()

for run_id in run_ids:
    print(run_id)
    outputs_file = glob(f'outputs/toxicity/llm/{run_id}/outputs_epsilon*.txt')
    data = read_outputs(outputs_file[0])
    edited_indexes = data.loc[data['edited']==True].index.tolist()
    if len(common_edited_indexes) == 0:
        common_edited_indexes = set(edited_indexes)
    else:
        common_edited_indexes &= set(edited_indexes)
    

vzdu2wd8
wwhu40zw
4hp0i0eu
9ldaxvli
ny9pw5ce
48bqsqpf


In [27]:
common_edited_indexes = sorted(list(common_edited_indexes))

In [28]:
with open('new_module/data/toxicity-avoidance/toxicity_epsilon_ablation_index.txt', 'w') as f:
    f.write(' '.join([str(i) for i in common_edited_indexes]))
    f.write('\n')

### Consistency (Save index)

In [29]:
# list of run ids
run_ids = ["e6xxarvb", 
           "tk10yknt",
           "6nccxn2c",
           "9oike8f1",
           "uqd41k9e",
           "rsxj4efi"]

In [34]:
common_edited_indexes = set()

for run_id in run_ids:
    print(run_id)
    outputs_file = glob(f'outputs/nli/{run_id}/outputs_epsilon*.txt')
    outputs_file = [x for x in outputs_file if 'results' not in x]
    print(outputs_file)
    data = read_outputs(outputs_file[0])
    edited_indexes = data.loc[data['edited']==True].index.tolist()
    if len(common_edited_indexes) == 0:
        common_edited_indexes = set(edited_indexes)
    else:
        common_edited_indexes &= set(edited_indexes)
    

e6xxarvb
['outputs/nli/e6xxarvb/outputs_epsilon0.5.txt']
tk10yknt
['outputs/nli/tk10yknt/outputs_epsilon0.6.txt']
6nccxn2c
['outputs/nli/6nccxn2c/outputs_epsilon0.7.txt']
9oike8f1
['outputs/nli/9oike8f1/outputs_epsilon0.8.txt']
uqd41k9e
['outputs/nli/uqd41k9e/outputs_epsilon0.9.txt']
rsxj4efi
['outputs/nli/rsxj4efi/outputs_epsilon0.99.txt']


In [35]:
common_edited_indexes = sorted(list(common_edited_indexes))

In [37]:
with open('new_module/data/logical-consistency/anli-r2-test_prompt_4_epsilon_ablation_index.txt', 'w') as f:
    f.write(' '.join([str(i) for i in common_edited_indexes]))
    f.write('\n')

### Toxicity (Aggregate common index only results)

In [17]:
# list of run ids
run_ids = ["vzdu2wd8", 
           "wwhu40zw",
           "4hp0i0eu",
           "9ldaxvli",
           "ny9pw5ce",
           "48bqsqpf"]
epsilons = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

In [20]:
results = []
for epsilon, run_id in zip(epsilons, run_ids):
    df = pd.read_csv(f'outputs/toxicity/llm/{run_id}/outputs_epsilon{epsilon}.txt-results_epsilon_ablation_index.csv')
    df['epsilon'] = epsilon
    results.append(df)
results = pd.concat(results)

In [21]:
results

,nicknames,sbert,sbert_count,sbert_ratio,avg_max_toxicity,avg_toxicity,toxic_proba,toxic_75_proba,ppl_qwen,total_ppl_qwen,delta_ppl,fluency_metrics,dist-3,rep_rate,num_edits,epsilon
0,llm_gens_epsilon_ablation_index,0.820967,41,1.0,0.179748,0.132139,0.0,0.0,7.929687,7.181441,NaN,0.95122,0.889957,0,41,0.50
0,llm_gens_epsilon_ablation_index,0.820967,41,1.0,0.179561,0.132073,0.0,0.0,7.929687,7.181441,NaN,0.95122,0.889957,0,41,0.60
0,llm_gens_epsilon_ablation_index,0.820967,41,1.0,0.180250,0.132084,0.0,0.0,7.929687,7.181441,NaN,0.95122,0.889957,0,41,0.70
0,llm_gens_epsilon_ablation_index,0.820967,41,1.0,0.179667,0.132198,0.0,0.0,7.929687,7.181441,NaN,0.95122,0.889957,0,41,0.80
0,llm_gens_epsilon_ablation_index,0.820967,41,1.0,0.180464,0.132077,0.0,0.0,7.929687,7.181441,NaN,0.95122,0.889957,0,41,0.90
0,llm_gens_epsilon_ablation_index,0.820967,41,1.0,0.179667,0.131989,0.0,0.0,7.929687,7.181441,NaN,0.95122,0.889957,0,41,0.95


In [ ]:
results.to_csv('new_module/_notebooks/outputs/epsilon_ablation/toxicity_epsilon_ablation.csv',index=False)

In [ ]:
## 거의 identical 한 결과가 리턴됨. (toxicity 관련 metric 제외하면 모두 동일함. 아예 값 자체가 똑같을 거라는 의심이 든다.)
## 기대한 것은 공통적으로 고친 sample일지라도 epsilon을 어떻게
## 설정했느냐에 따라서 epsilon을 높게 설정했으면 early stop 까지 오래 걸리니까 더 많이 고칠 거라고 생각했는데,
## 착각이었던 게 n_iter=1이니까 어짜피 early stop 은 irrelevant함.
## 1회씩만 고칠 때 영향을 줄 수 있는 부분은, best candidate을 고를 때
## epsilon을 만족한 것 중에서 가장 fluent한 것을 고르는 부분에서 (allsat_primary)
## epsilon이 더 높으면, fluency보다 epsilon satisfaction을 우선시 하게 되므로 toxicity는 떨어지고 fluency도 떨어질 것으로 예상된다.
## 다만 epsilon을 만족하는 옵션이 없을 경우에는 fluency 와 constraint satisfaction의 weighted sum 으로 고르기 때문에 
## 그렇게 되면 epsilon의 선택에 영향을 받지 않는다. epsilon이 높을 수록 이 default mechanism으로 fall back 하는 샘플이 많아질 것 같다.

### Consistency (Aggregate common index only results)

In [11]:
# list of run ids
run_ids = ["e6xxarvb", 
           "tk10yknt",
           "6nccxn2c",
           "9oike8f1",
           "uqd41k9e",
           "rsxj4efi"]
epsilons = [0.5, 0.6, 0.7, 0.8, 0.9, 0.99]

In [14]:
results = []
for epsilon, run_id in zip(epsilons, run_ids):
    df = pd.read_csv(f'outputs/nli/{run_id}/outputs_epsilon{epsilon}.txt-results_epsilon_ablation_index.csv')
    df['epsilon'] = epsilon
    results.append(df)
results = pd.concat(results)

In [15]:
results

,nicknames,sbert,sbert_count,sbert_ratio,contra_prob,ppl,total_ppl,delta_ppl,fluency_metrics,dist-3,rep_rate,num_edits,epsilon
0,llm_gens_epsilon_ablation_index,0.671917,872,0.841699,0.136100,26.910005,17.245189,NaN,0.964286,0.739154,0,1036,0.50
0,llm_gens_epsilon_ablation_index,0.671112,871,0.840734,0.130309,27.587567,17.363954,NaN,0.963320,0.738572,0,1036,0.60
0,llm_gens_epsilon_ablation_index,0.673758,874,0.843629,0.111004,27.460850,17.613311,NaN,0.952703,0.738571,0,1036,0.70
0,llm_gens_epsilon_ablation_index,0.672394,868,0.837838,0.102317,28.916670,18.199181,NaN,0.950772,0.738214,0,1036,0.80
0,llm_gens_epsilon_ablation_index,0.672709,870,0.839768,0.101351,29.587474,18.383126,NaN,0.951737,0.736708,0,1036,0.90
0,llm_gens_epsilon_ablation_index,0.674416,872,0.841699,0.098456,30.969869,18.761097,NaN,0.959459,0.736998,0,1036,0.99


In [ ]:
results.to_csv('new_module/_notebooks/outputs/epsilon_ablation/nli_epsilon_ablation.csv',index=False)

In [ ]:
## epsilon이 높아질 수록, contra_prob은 낮아지고, total_ppl은 높아지는 양상이다.
## 더 sample수가 많아서 그런걸까? 예측과 유사하게 나왔다.